In [1]:
import sys

print("Python version:", sys.version)
print("Python location:", sys.executable)

Python version: 3.12.8 | packaged by Anaconda, Inc. | (main, Dec 11 2024, 16:48:34) [MSC v.1929 64 bit (AMD64)]
Python location: c:\Users\laksh\anaconda3\python.exe


In [2]:
%pip install pypdf sentence-transformers chromadb

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pypdf
import chromadb
from sentence_transformers import SentenceTransformer

print("Everything installed successfully!")

Everything installed successfully!


In [4]:
with open("documents/rag_sample.txt", "r", encoding="utf-8") as file:
    text = file.read()

print(text[:1500])

Company: NovaRetail Analytics
Document: 2026 Business Performance Summary

1. Company Overview
NovaRetail Analytics is a fictional retail technology company that provides data analytics tools to supermarkets and online retailers. The company focuses on sales forecasting, customer segmentation, inventory planning, and dashboard reporting.

2. Revenue Performance
In 2026, NovaRetail Analytics reported total revenue of AUD 48 million. This represented a 20% increase compared with 2025. The strongest growth came from the cloud analytics subscription business, which contributed AUD 28 million in revenue.

3. Customer Growth
The company served 320 business customers in 2026. This was an increase from 250 customers in 2025. Most new customers were medium-sized retail businesses in Australia and New Zealand.

4. Product Performance
The company's most popular product was InsightCloud, a subscription-based analytics platform. InsightCloud generated 58% of total company revenue. A second product,

In [5]:
print("Total characters:", len(text))
print("Total words:", len(text.split()))

Total characters: 2819
Total words: 396


In [6]:
import re

clean_text = re.sub(r"\s+", " ", text).strip()

print(clean_text[:1000])

Company: NovaRetail Analytics Document: 2026 Business Performance Summary 1. Company Overview NovaRetail Analytics is a fictional retail technology company that provides data analytics tools to supermarkets and online retailers. The company focuses on sales forecasting, customer segmentation, inventory planning, and dashboard reporting. 2. Revenue Performance In 2026, NovaRetail Analytics reported total revenue of AUD 48 million. This represented a 20% increase compared with 2025. The strongest growth came from the cloud analytics subscription business, which contributed AUD 28 million in revenue. 3. Customer Growth The company served 320 business customers in 2026. This was an increase from 250 customers in 2025. Most new customers were medium-sized retail businesses in Australia and New Zealand. 4. Product Performance The company's most popular product was InsightCloud, a subscription-based analytics platform. InsightCloud generated 58% of total company revenue. A second product, Sto

In [7]:
chunk_size = 500

chunks = []

for i in range(0, len(clean_text), chunk_size):
    chunk = clean_text[i:i + chunk_size]
    chunks.append(chunk)

print("Number of chunks:", len(chunks))

Number of chunks: 6


In [8]:
print("CHUNK 1:")
print(chunks[0])

print("\nCHUNK 2:")
print(chunks[1])

CHUNK 1:
Company: NovaRetail Analytics Document: 2026 Business Performance Summary 1. Company Overview NovaRetail Analytics is a fictional retail technology company that provides data analytics tools to supermarkets and online retailers. The company focuses on sales forecasting, customer segmentation, inventory planning, and dashboard reporting. 2. Revenue Performance In 2026, NovaRetail Analytics reported total revenue of AUD 48 million. This represented a 20% increase compared with 2025. The strongest 

CHUNK 2:
growth came from the cloud analytics subscription business, which contributed AUD 28 million in revenue. 3. Customer Growth The company served 320 business customers in 2026. This was an increase from 250 customers in 2025. Most new customers were medium-sized retail businesses in Australia and New Zealand. 4. Product Performance The company's most popular product was InsightCloud, a subscription-based analytics platform. InsightCloud generated 58% of total company revenue. A

In [9]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
embeddings = model.encode(chunks)

print("Number of chunks:", len(chunks))
print("Embeddings shape:", embeddings.shape)

Number of chunks: 6
Embeddings shape: (6, 384)


In [11]:
print(embeddings[0][:20])

[-0.04732606 -0.00832066 -0.08104049 -0.00574777 -0.07062984 -0.02525331
 -0.02541976 -0.05195868 -0.00483983  0.01856504 -0.09471542  0.01139946
 -0.02269418 -0.00094117 -0.12920628 -0.03701599  0.04035267 -0.01086608
  0.01530766 -0.03875718]


In [12]:
chunks[0]

'Company: NovaRetail Analytics Document: 2026 Business Performance Summary 1. Company Overview NovaRetail Analytics is a fictional retail technology company that provides data analytics tools to supermarkets and online retailers. The company focuses on sales forecasting, customer segmentation, inventory planning, and dashboard reporting. 2. Revenue Performance In 2026, NovaRetail Analytics reported total revenue of AUD 48 million. This represented a 20% increase compared with 2025. The strongest '

In [13]:
embeddings[0]

array([-4.73260619e-02, -8.32066033e-03, -8.10404867e-02, -5.74776856e-03,
       -7.06298351e-02, -2.52533127e-02, -2.54197605e-02, -5.19586764e-02,
       -4.83983429e-03,  1.85650438e-02, -9.47154164e-02,  1.13994628e-02,
       -2.26941779e-02, -9.41165490e-04, -1.29206285e-01, -3.70159931e-02,
        4.03526723e-02, -1.08660804e-02,  1.53076565e-02, -3.87571827e-02,
       -4.35095504e-02,  6.69215322e-02,  3.99224367e-03, -1.15939956e-02,
        3.81248370e-02,  5.79116978e-02, -8.90825093e-02,  2.70379819e-02,
       -3.01649664e-02, -5.57375066e-02, -1.35017699e-02,  3.19152810e-02,
       -4.51806188e-02,  7.63959661e-02, -7.33915344e-02, -3.26474570e-02,
       -7.09659187e-03,  1.41819948e-02, -3.62636074e-02, -6.45054951e-02,
        7.63433613e-03, -3.74873690e-02, -4.12065238e-02,  8.13252525e-04,
        3.40331234e-02, -8.13588616e-04, -9.42125395e-02,  1.04910610e-02,
       -3.83376330e-03,  7.75918141e-02, -9.81849283e-02, -8.46878961e-02,
       -8.67172703e-03, -

In [14]:
query = "What was the company's revenue in 2026?"

In [15]:
query_embedding = model.encode([query])

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(query_embedding, embeddings)[0]

print(similarities)

[0.42338353 0.44213265 0.32151353 0.02565175 0.21336076 0.3733865 ]


In [17]:
best_index = similarities.argmax()

print("Best matching chunk:")
print(chunks[best_index])

print("\nSimilarity score:")
print(similarities[best_index])

Best matching chunk:
growth came from the cloud analytics subscription business, which contributed AUD 28 million in revenue. 3. Customer Growth The company served 320 business customers in 2026. This was an increase from 250 customers in 2025. Most new customers were medium-sized retail businesses in Australia and New Zealand. 4. Product Performance The company's most popular product was InsightCloud, a subscription-based analytics platform. InsightCloud generated 58% of total company revenue. A second product, Sto

Similarity score:
0.44213265


In [18]:
import numpy as np

top_k = 3

top_indices = np.argsort(similarities)[::-1][:top_k]

print("Top matching chunks:\n")

for rank, index in enumerate(top_indices, start=1):
    print(f"Rank {rank}")
    print(f"Similarity score: {similarities[index]:.4f}")
    print(chunks[index])
    print("-" * 60)

Top matching chunks:

Rank 1
Similarity score: 0.4421
growth came from the cloud analytics subscription business, which contributed AUD 28 million in revenue. 3. Customer Growth The company served 320 business customers in 2026. This was an increase from 250 customers in 2025. Most new customers were medium-sized retail businesses in Australia and New Zealand. 4. Product Performance The company's most popular product was InsightCloud, a subscription-based analytics platform. InsightCloud generated 58% of total company revenue. A second product, Sto
------------------------------------------------------------
Rank 2
Similarity score: 0.4234
Company: NovaRetail Analytics Document: 2026 Business Performance Summary 1. Company Overview NovaRetail Analytics is a fictional retail technology company that provides data analytics tools to supermarkets and online retailers. The company focuses on sales forecasting, customer segmentation, inventory planning, and dashboard reporting. 2. Revenue Pe

In [19]:
import chromadb

client = chromadb.Client()

collection = client.get_or_create_collection(
    name="rag_demo"
)

print("Collection created!")

Collection created!


In [20]:
ids = [f"chunk_{i}" for i in range(len(chunks))]

collection.add(
    ids=ids,
    documents=chunks,
    embeddings=embeddings.tolist()
)

print("Number of stored chunks:", collection.count())

Number of stored chunks: 6


In [21]:
query = "What was the company's revenue in 2026?"

query_embedding = model.encode([query])

In [22]:
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=3
)

print(results["documents"][0])

["growth came from the cloud analytics subscription business, which contributed AUD 28 million in revenue. 3. Customer Growth The company served 320 business customers in 2026. This was an increase from 250 customers in 2025. Most new customers were medium-sized retail businesses in Australia and New Zealand. 4. Product Performance The company's most popular product was InsightCloud, a subscription-based analytics platform. InsightCloud generated 58% of total company revenue. A second product, Sto", 'Company: NovaRetail Analytics Document: 2026 Business Performance Summary 1. Company Overview NovaRetail Analytics is a fictional retail technology company that provides data analytics tools to supermarkets and online retailers. The company focuses on sales forecasting, customer segmentation, inventory planning, and dashboard reporting. 2. Revenue Performance In 2026, NovaRetail Analytics reported total revenue of AUD 48 million. This represented a 20% increase compared with 2025. The stro

In [23]:
for rank, document in enumerate(results["documents"][0], start=1):
    print(f"Rank {rank}")
    print(document)
    print("-" * 60)

Rank 1
growth came from the cloud analytics subscription business, which contributed AUD 28 million in revenue. 3. Customer Growth The company served 320 business customers in 2026. This was an increase from 250 customers in 2025. Most new customers were medium-sized retail businesses in Australia and New Zealand. 4. Product Performance The company's most popular product was InsightCloud, a subscription-based analytics platform. InsightCloud generated 58% of total company revenue. A second product, Sto
------------------------------------------------------------
Rank 2
Company: NovaRetail Analytics Document: 2026 Business Performance Summary 1. Company Overview NovaRetail Analytics is a fictional retail technology company that provides data analytics tools to supermarkets and online retailers. The company focuses on sales forecasting, customer segmentation, inventory planning, and dashboard reporting. 2. Revenue Performance In 2026, NovaRetail Analytics reported total revenue of AUD 48

In [24]:
%pip install rank-bm25

Note: you may need to restart the kernel to use updated packages.


In [25]:
from rank_bm25 import BM25Okapi

tokenized_chunks = [chunk.lower().split() for chunk in chunks]

bm25 = BM25Okapi(tokenized_chunks)

print("BM25 ready!")

BM25 ready!


In [26]:
query = "revenue 2026"

tokenized_query = query.lower().split()

bm25_scores = bm25.get_scores(tokenized_query)

print(bm25_scores)

[2.14542347 0.         0.         0.         0.         0.69335367]


In [27]:
import numpy as np

top_bm25_indices = np.argsort(bm25_scores)[::-1][:3]

for rank, index in enumerate(top_bm25_indices, start=1):
    print(f"Rank {rank}")
    print(f"BM25 score: {bm25_scores[index]:.4f}")
    print(chunks[index])
    print("-" * 60)

Rank 1
BM25 score: 2.1454
Company: NovaRetail Analytics Document: 2026 Business Performance Summary 1. Company Overview NovaRetail Analytics is a fictional retail technology company that provides data analytics tools to supermarkets and online retailers. The company focuses on sales forecasting, customer segmentation, inventory planning, and dashboard reporting. 2. Revenue Performance In 2026, NovaRetail Analytics reported total revenue of AUD 48 million. This represented a 20% increase compared with 2025. The strongest 
------------------------------------------------------------
Rank 2
BM25 score: 0.6934
 tool for finance, technical support, and consulting teams. 10. Summary NovaRetail Analytics grew strongly in 2026, with revenue increasing to AUD 48 million and customer numbers reaching 320. The company is now investing in AI, including a retrieval-augmented generation system for internal document search.
------------------------------------------------------------
Rank 3
BM25 scor

In [28]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(chunks)

print("Embeddings ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings ready!


In [29]:
query = "What are the company's plans for Singapore in 2027?"

In [30]:
from sklearn.metrics.pairwise import cosine_similarity

query_embedding = model.encode([query])

semantic_scores = cosine_similarity(
    query_embedding,
    embeddings
)[0]

In [31]:
tokenized_query = query.lower().split()

bm25_scores = bm25.get_scores(tokenized_query)

In [32]:
import numpy as np

def normalize(scores):
    scores = np.array(scores, dtype=float)

    if scores.max() == scores.min():
        return np.zeros_like(scores)

    return (scores - scores.min()) / (scores.max() - scores.min())

In [33]:
semantic_normalized = normalize(semantic_scores)
bm25_normalized = normalize(bm25_scores)

In [34]:
hybrid_scores = (
    0.6 * semantic_normalized +
    0.4 * bm25_normalized
)

In [35]:
top_hybrid_indices = np.argsort(hybrid_scores)[::-1][:3]

for rank, index in enumerate(top_hybrid_indices, start=1):
    print(f"Rank {rank}")
    print(f"Semantic: {semantic_normalized[index]:.4f}")
    print(f"BM25: {bm25_normalized[index]:.4f}")
    print(f"Hybrid: {hybrid_scores[index]:.4f}")
    print(chunks[index])
    print("-" * 60)

Rank 1
Semantic: 1.0000
BM25: 1.0000
Hybrid: 1.0000
nt with each answer 8. Risk and Governance The company requires the AI system to protect confidential information. Access controls must ensure employees only retrieve documents they are authorised to view. The company also wants responses to be grounded in source documents to reduce hallucinations. 9. Future Plans In 2027, NovaRetail Analytics plans to expand into Singapore and increase its AI engineering team. The company expects the internal document search project to become a key productivity
------------------------------------------------------------
Rank 2
Semantic: 0.9003
BM25: 0.1774
Hybrid: 0.6111
ckSense, helped customers predict inventory demand and reduce stockouts. 5. Operational Challenges NovaRetail Analytics experienced three major challenges in 2026: - Higher cloud infrastructure costs - Difficulty hiring experienced data engineers - Increased competition from international analytics providers 6. Technology Strategy T

In [36]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Reranker ready!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranker ready!


In [37]:
candidate_chunks = [chunks[i] for i in top_hybrid_indices]

for i, chunk in enumerate(candidate_chunks, start=1):
    print(f"Candidate {i}")
    print(chunk)
    print("-" * 60)

Candidate 1
nt with each answer 8. Risk and Governance The company requires the AI system to protect confidential information. Access controls must ensure employees only retrieve documents they are authorised to view. The company also wants responses to be grounded in source documents to reduce hallucinations. 9. Future Plans In 2027, NovaRetail Analytics plans to expand into Singapore and increase its AI engineering team. The company expects the internal document search project to become a key productivity
------------------------------------------------------------
Candidate 2
ckSense, helped customers predict inventory demand and reduce stockouts. 5. Operational Challenges NovaRetail Analytics experienced three major challenges in 2026: - Higher cloud infrastructure costs - Difficulty hiring experienced data engineers - Increased competition from international analytics providers 6. Technology Strategy The company plans to invest in artificial intelligence and retrieval-augmented ge

In [38]:
query_chunk_pairs = [
    [query, chunk]
    for chunk in candidate_chunks
]

rerank_scores = reranker.predict(query_chunk_pairs)

print(rerank_scores)

[ 6.580902  -2.653449  -4.7635217]


In [39]:
reranked_indices = np.argsort(rerank_scores)[::-1]

print("Reranked results:\n")

for rank, idx in enumerate(reranked_indices, start=1):
    print(f"Rank {rank}")
    print(f"Reranker score: {rerank_scores[idx]:.4f}")
    print(candidate_chunks[idx])
    print("-" * 60)

Reranked results:

Rank 1
Reranker score: 6.5809
nt with each answer 8. Risk and Governance The company requires the AI system to protect confidential information. Access controls must ensure employees only retrieve documents they are authorised to view. The company also wants responses to be grounded in source documents to reduce hallucinations. 9. Future Plans In 2027, NovaRetail Analytics plans to expand into Singapore and increase its AI engineering team. The company expects the internal document search project to become a key productivity
------------------------------------------------------------
Rank 2
Reranker score: -2.6534
ckSense, helped customers predict inventory demand and reduce stockouts. 5. Operational Challenges NovaRetail Analytics experienced three major challenges in 2026: - Higher cloud infrastructure costs - Difficulty hiring experienced data engineers - Increased competition from international analytics providers 6. Technology Strategy The company plans to inve

In [40]:
%pip install sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [41]:
best_chunks = [
    candidate_chunks[idx]
    for idx in reranked_indices
]

context = "\n\n".join(best_chunks)

print(context)

nt with each answer 8. Risk and Governance The company requires the AI system to protect confidential information. Access controls must ensure employees only retrieve documents they are authorised to view. The company also wants responses to be grounded in source documents to reduce hallucinations. 9. Future Plans In 2027, NovaRetail Analytics plans to expand into Singapore and increase its AI engineering team. The company expects the internal document search project to become a key productivity

ckSense, helped customers predict inventory demand and reduce stockouts. 5. Operational Challenges NovaRetail Analytics experienced three major challenges in 2026: - Higher cloud infrastructure costs - Difficulty hiring experienced data engineers - Increased competition from international analytics providers 6. Technology Strategy The company plans to invest in artificial intelligence and retrieval-augmented generation systems. One planned project will allow employees to search internal financ

In [43]:
prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""

In [81]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Generator ready!")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

c:\Users\laksh\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\laksh\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\laksh\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-syste

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generator ready!


In [82]:
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
)

outputs = generator_model.generate(
    **inputs,
    max_new_tokens=100
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Question:", query)
print("\nAnswer:", answer)

Question: What are the company's plans for Singapore in 2027?

Answer: expand into Singapore


I built a simple RAG pipeline in Python. I loaded and cleaned a text document, split it into chunks, 
created sentence-transformer embeddings, performed semantic retrieval and BM25 keyword search, 
combined them using hybrid search, reranked the results, and passed the retrieved context to a small language model to generate a grounded answer.”
